In [6]:
from dotenv import load_dotenv
import requests
from agents import Agent, Runner, trace, function_tool, ModelSettings
from openai.types.responses import ResponseTextDeltaEvent
import os
import asyncio
import smtplib
from email.message import EmailMessage
load_dotenv(override=True)
MODEL_NAME = "gpt-5.4-mini"

In [7]:
EMAIL_ADDRESS = os.getenv("EMAIL_ADDRESS")
EMAIL_SMTP_SERVER = os.getenv("EMAIL_SMTP_SERVER")
EMAIL_APP_PASSWORD = os.getenv("EMAIL_APP_PASSWORD")

if EMAIL_ADDRESS:
    print("Email address is set")
else:
    print("Email address is not set")

if EMAIL_SMTP_SERVER:
    print("SMTP server is set")
else:
    print("SMTP server is not set")

if EMAIL_APP_PASSWORD:
    print("App password is set")
else:
    print("App password is not set")

USE_EMAIL = EMAIL_ADDRESS and EMAIL_SMTP_SERVER and EMAIL_APP_PASSWORD

if USE_EMAIL:
    print("Email is set up and we will try using it")
else:
    print("Email is not set up; we will send push notifications instead")

Email address is set
SMTP server is set
App password is set
Email is set up and we will try using it


In [8]:
# Here we go

def send_email(subject, text_body, html_body):
    msg = EmailMessage()
    msg["From"] = EMAIL_ADDRESS
    msg["To"] = EMAIL_ADDRESS
    msg["Subject"] = subject
    msg.set_content(text_body)
    msg.add_alternative(html_body, subtype="html")

    with smtplib.SMTP(EMAIL_SMTP_SERVER, 587) as server:
        server.starttls()
        server.login(EMAIL_ADDRESS, EMAIL_APP_PASSWORD)
        server.send_message(msg)

In [5]:
send_email("Testing testing 123", "Fingers crossed..", "<html><body><strong>Fingers</strong> crossed..</body></html>")

In [9]:
intro = """ 
You are a sales agent working for ComplAI,
a company that provides a SaaS tool for ensuring SOC2 compliance and preparing audits, powered by AI.
You write emails.
"""
instructions1 = intro + "Your email style is professional, serious, with gravitas and credibility"
instructions2 = intro + "Your email style is witty, engaging, and humorous."
instructions3 = intro + "Your email style is concise, to the point, in the style of a busy senior executive."

In [10]:
sales_agent1 = Agent(name="Professional Sales Agent", instructions=instructions1, model=MODEL_NAME)
sales_agent2 = Agent(name="Humorous Sales Agent", instructions=instructions2, model=MODEL_NAME)
sales_agent3 = Agent(name="Executive Sales Agent", instructions=instructions3, model=MODEL_NAME)

In [11]:
result = Runner.run_streamed(sales_agent1, input="Write a cold sales email")
async for event in result.stream_events():
    if event.type == "raw_response_event" and isinstance(event.data, ResponseTextDeltaEvent):
        print(event.data.delta, end="", flush=True)

Subject: Preparing for SOC 2 without slowing the business

Hi [First Name],

Many teams know SOC 2 is necessary, but the real challenge is getting audit-ready without creating weeks of manual work across engineering, security, and operations.

ComplAI helps companies prepare for SOC 2 faster with an AI-powered platform that streamlines evidence collection, tracks controls, and keeps audit readiness visible in one place.

If you’re planning a SOC 2 effort this quarter, I’d welcome a brief conversation to see whether we can help reduce the burden on your team.

Best,  
[Your Name]  
ComplAI

In [12]:
message = "Write a cold sales email"

with trace("Parallel cold emails"):
    results  = await asyncio.gather(
        Runner.run(sales_agent1, message),
        Runner.run(sales_agent2, message),
        Runner.run(sales_agent3, message)
    )
    outputs = [result.final_output for result in results]

    emails = "Cold sales emails:\n\n" + "\n\nEmail:\n\n".join(outputs)

In [13]:
decision = """
You pick the best cold sales email from the given options.
Imagine you are a customer and pick the one you are most likely to respond to.
Do not give and explanation; reply with the second email only.
"""

sales_picker = Agent(name="sales_picker", instructions=decision, model=MODEL_NAME)
best = await Runner.run(sales_picker, emails)
print(f"Best sales email:\n{best.final_output}")

Best sales email:
Email:

Subject: SOC 2 prep without the scramble

Hi [Name],

If SOC 2 compliance is on your roadmap, ComplAI helps teams get audit-ready faster with AI-assisted evidence collection, control mapping, and audit prep.

Instead of chasing screenshots and spreadsheets, your team gets a clear workflow to stay organized and reduce manual work.

Open to a quick chat next week to see if this could help [Company]?

Best,
[Your Name]


In [14]:
@function_tool
def send_email_tool(subject: str, text_body: str, html_body:str) -> str:
    """
    Send out an email with the given subject and body to all sells prospects

    Args:
        subject: The subject of the email
        text_body: The body of the email as plain text
        html_body: The HTML body of the email
    """
    send_email(subject, text_body, html_body)
    return "Email sent successfully"

In [15]:
decision = """
You pick the best cold sales email from the given option.
Imagine you are a customer and pick the one you are most likely to respond to.
Then use your tool to send the email
"""

require_tool = ModelSettings(tool_choice="required")
sales_sender = Agent(name="Sales_Sender", instructions=decision, model=MODEL_NAME, tools=[send_email_tool], model_settings=require_tool)

In [16]:
message = "Write a cold sales email"

with trace("Sales selection workflow with sending"):
    results = await asyncio.gather(
        Runner.run(sales_agent1, message),
        Runner.run(sales_agent2, message),
        Runner.run(sales_agent3, message)
    )
    outputs = [result.final_output for result in results]
    emails = "Cold sales emails:\n\n" + "\n\nEmail:\n\n".join(outputs)

    response = await Runner.run(sales_sender, emails)

    print(f"Final response:\n{response.final_output}")

Final response:
Done — I picked and sent the third email (“SOC 2 prep without the usual scramble”).
